# Mục 3d — Trường hợp 2 Giảm số dòng bằng tổng hợp nhóm
**Võ Minh Huy — 2311110117**

Tổng hợp bản đầy đủ 299 hồ sơ sau xử lý nhãn theo nhóm tuổi và sex. Lưu số lượng và mean EF, tiểu cầu, creatinine. Không dùng bảng giảm chiều ở trường hợp 1 vì bảng đó không có sex.

## 1. Chuẩn bị nhóm tuổi và giới tính
Tuổi sau N02; sex sau N01, có dùng giá trị đã điền. A1 dưới 50; A2 [50,60); A3 [60,70); A4 từ 70. Không dùng nhãn DEATH_EVENT nên vẫn đại diện hồ sơ chờ xác minh nhãn.

In [1]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP, localcontext
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_vao = thu_muc / "ket_qua/15_khong_nhat_quan/02_death_event/ban_lam_viec_nhan_python.csv"
ban_day_du = pd.read_csv(file_vao, dtype=str, keep_default_na=False)
cot_can = ["age", "sex", "ejection_fraction", "platelets", "serum_creatinine"]
ban_nhom = ban_day_du[cot_can].copy()
for cot in cot_can:
    ban_nhom[cot] = ban_nhom[cot].map(lambda x: Decimal(x) if x.strip() else None)
if ban_nhom["age"].isna().any() or any(x <= 0 for x in ban_nhom["age"]):
    raise ValueError("Cần tuổi hợp lệ trước tổng hợp.")
if not ban_nhom["sex"].isin([0, 1]).all():
    raise ValueError("Cần sex 0/1 trước tổng hợp.")

def nhom_tuoi(tuoi):
    if tuoi < 50:
        return "A1"
    if tuoi < 60:
        return "A2"
    if tuoi < 70:
        return "A3"
    return "A4"

ban_nhom["NhomTuoi"] = ban_nhom["age"].map(nhom_tuoi)
ban_nhom["sex"] = ban_nhom["sex"].map(int)
pd.DataFrame({"SoHoSoDauVao":[len(ban_nhom)]})

,SoHoSoDauVao
0,299


## 2. Tổng hợp và kiểm tra số hồ sơ
GROUP BY nhóm tuổi và sex. Mean bằng tổng giá trị không thiếu chia số giá trị không thiếu. Lưu 10 số lẻ, làm tròn 2 số chỉ ở bảng hiển thị.

In [2]:
def mean_chinh_xac(cot):
    gia_tri = cot.dropna().tolist()
    if not gia_tri:
        return None
    with localcontext() as ctx:
        ctx.prec = 38
        return (sum(gia_tri, Decimal(0)) / len(gia_tri)).quantize(
            Decimal("0.0000000001"), rounding=ROUND_HALF_UP)

dong_nhom = []
for (tuoi, sex), nhom in ban_nhom.groupby(["NhomTuoi", "sex"], sort=True):
    dong_nhom.append({"NhomTuoi":tuoi, "sex":sex, "SoHoSo":len(nhom),
        "EFMean":mean_chinh_xac(nhom["ejection_fraction"]),
        "PlateletsMean":mean_chinh_xac(nhom["platelets"]),
        "CreatinineMean":mean_chinh_xac(nhom["serum_creatinine"])})
nhom_tong_hop = pd.DataFrame(dong_nhom)
so_truoc = len(ban_day_du)
so_sau = len(nhom_tong_hop)
so_dai_dien = int(nhom_tong_hop["SoHoSo"].sum())
ty_le = (Decimal(100) * (so_truoc - so_sau) / so_truoc).quantize(
    Decimal("0.01"), rounding=ROUND_HALF_UP)
tong_ket = pd.DataFrame({"SoDongTruoc":[so_truoc], "SoDongTongHop":[so_sau],
    "TongHoSoDaiDien":[so_dai_dien], "TyLeGiamPhanTram":[ty_le],
    "KiemTra":["Đủ hồ sơ" if so_dai_dien == so_truoc else "Cần kiểm tra"]})
tong_ket

,SoDongTruoc,SoDongTongHop,TongHoSoDaiDien,TyLeGiamPhanTram,KiemTra
0,299,8,299,97.32,Đủ hồ sơ


## 3. Bảng kết quả tám nhóm
Một dòng đại diện một nhóm, không còn là một bệnh nhân. sex 0 = nữ, 1 = nam.

In [3]:
khoang_tuoi = {"A1":"Dưới 50", "A2":"50 đến dưới 60",
               "A3":"60 đến dưới 70", "A4":"Từ 70 trở lên"}
hien_thi = nhom_tong_hop.copy()
hien_thi["KhoangTuoi"] = hien_thi["NhomTuoi"].map(khoang_tuoi)
hien_thi["GioiTinh"] = hien_thi["sex"].map({0:"Nữ", 1:"Nam"})
for goc, moi in [("EFMean", "EFTrungBinh"), ("PlateletsMean", "TieuCauTrungBinh"),
                  ("CreatinineMean", "CreatinineTrungBinh")]:
    hien_thi[moi] = hien_thi[goc].map(lambda v: v.quantize(
        Decimal("0.01"), rounding=ROUND_HALF_UP) if v is not None else None)
hien_thi = hien_thi[["NhomTuoi", "KhoangTuoi", "sex", "GioiTinh", "SoHoSo",
                     "EFTrungBinh", "TieuCauTrungBinh", "CreatinineTrungBinh"]]
hien_thi

,NhomTuoi,KhoangTuoi,sex,GioiTinh,SoHoSo,EFTrungBinh,TieuCauTrungBinh,CreatinineTrungBinh
0,A1,Dưới 50,0,Nữ,18,40.33,278170.78,1.15
1,A1,Dưới 50,1,Nam,24,35.71,260437.65,1.14
2,A2,50 đến dưới 60,0,Nữ,27,37.37,282111.11,1.43
3,A2,50 đến dưới 60,1,Nam,50,38.12,259512.91,1.20
4,A3,60 đến dưới 70,0,Nữ,38,40.00,249378.68,1.67
5,A3,60 đến dưới 70,1,Nam,73,37.05,263922.15,1.52
6,A4,Từ 70 trở lên,0,Nữ,22,41.09,311415.76,1.37
7,A4,Từ 70 trở lên,1,Nam,47,37.45,243673.86,1.43


**Nhận xét:** 299 dòng hồ sơ được biểu diễn bằng 8 dòng nhóm, giảm 97,32% số dòng. Số hồ sơ mỗi nhóm là 18, 24, 27, 50, 38, 73, 22, 47; tổng bằng 299. Bảng giữ count và mean, không giữ chi tiết từng bệnh nhân hoặc phân bố trong nhóm. Không thể khôi phục hồ sơ cá nhân từ bảng này. Khi gộp tiếp cần dùng số lượng làm trọng số, không lấy trung bình đơn giản của các mean. Dữ liệu đầy đủ vẫn giữ nguyên.

## 4. Lưu kết quả riêng
Lưu cả bảng mean 10 số lẻ và bảng hiển thị hai số lẻ; không thay thế bảng đầy đủ.

In [4]:
thu_muc_kq = thu_muc / "ket_qua/16_giam_thieu/02_tong_hop_nhom"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
tong_ket.to_csv(thu_muc_kq / "tong_ket_python.csv", index=False, encoding="utf-8-sig")
nhom_tong_hop.to_csv(thu_muc_kq / "nhom_python.csv", index=False, encoding="utf-8-sig")
hien_thi.to_csv(thu_muc_kq / "nhom_hien_thi_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu bảng tổng hợp tám nhóm; giữ nguyên dữ liệu đầy đủ."]})

,TrangThai
0,Đã lưu bảng tổng hợp tám nhóm; giữ nguyên dữ l...
